# 📊 Stage 1: Runtime Artifacts Creation – Overall Description

---

## 🎯 Objective
- Prepare all **precomputed artifacts** required for real-time recommendation.
- Convert heavy computations (co-occurrence, user behavior, context patterns) into **fast lookup structures**.
- Enable low-latency **runtime inference without retraining or recomputation**.

---

## 📂 Data Sources
- **Main Transaction Data (`main_data.csv`)**
  - Raw purchase logs (customer, item, category, time context)

- **Item Catalog**
  - Item metadata (itemId, itemName, category)

- **Category Rule Artifacts (JSON)**
  - Category relationships and family mappings

- **Category Embeddings**
  - Vector representations of categories

---

## ⚙️ Core Workflow

1. **Initialization**
   - Define file paths and output directory
   - Check data availability

2. **Data Loading**
   - Load main data, catalog, embeddings, and rules
   - Clean column names and formats

3. **Data Validation & Cleaning**
   - Ensure required columns exist
   - Convert types (IDs, dates, strings)
   - Derive missing features (e.g., weekOfMonth)

4. **Mapping Creation**
   - Build dictionaries:
     - `itemId → itemName`
     - `itemId → category`
     - `category → embedding vector`

5. **Feature Counter Construction**
   - Compute:
     - Item co-occurrence
     - Context-based item frequency
     - User-item interactions
     - User-category preferences
     - Category popularity

6. **Lookup Structure Creation**
   - Build fast-access structures:
     - `pair_to_related_items`
     - `context_to_items`
     - `user_to_items`

7. **User Behavior Modeling**
   - Determine default timeSlot per customer

8. **Artifact Serialization**
   - Save all structures as `.pkl` files

9. **Verification**
   - Check all files exist
   - Reload sample artifacts for validation

---

## 📦 Expected Outputs

### 📁 Pickle Artifacts
- **Counters**
  - `item_pair_counter.pkl`
  - `context_item_counter.pkl`
  - `user_item_counter.pkl`
  - `user_category_counter.pkl`
  - `category_popularity_counter.pkl`

- **Lookup Structures**
  - `pair_to_related_items.pkl`
  - `context_to_items.pkl`
  - `user_to_items.pkl`

- **Mappings**
  - `customer_default_timeslot.pkl`
  - `item_to_category.pkl`
  - `item_to_name.pkl`
  - `category_to_vector.pkl`
  - `category_family_map.pkl`

---

## 🔗 Role in Pipeline
- Acts as the **bridge between training and deployment**.
- Converts training-derived knowledge into **runtime-ready artifacts**.
- Enables:
  - Fast candidate generation
  - Real-time feature computation
  - Scalable recommendation serving

---

### 🚀 Summary
This stage transforms raw data and learned patterns into **efficient, reusable runtime components**, making the recommendation system:
- ⚡ Fast
- 📦 Lightweight
- 🚀 Deployment-ready  

<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: import ast
# Stage 1 Runtime Artifacts Setup – Code Overview

## 1. Explanation (Precise)
- Initializes paths for **runtime inference stage**.
- Loads references to:
  - Main data
  - Item catalog
  - Category rules
  - Category embeddings
- Creates output directory for runtime artifacts.
- Verifies existence of required files.
## 2. Possible Output
```bash
Main data exists: True
Item catalog exists: True
Category embedding exists: True
Output folder: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts

In [1]:
import ast
import pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")
DATA_DIR = BASE_DIR / "data"

MAIN_DATA_FILE = DATA_DIR / "main_data.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog_output" / "item_catalog.csv"
CATEGORY_RULE_FILE = DATA_DIR / "item_catalog_output" / "category_rule_artifacts.json"
CATEGORY_EMBEDDING_FILE = DATA_DIR / "basket_embedding_output" / "category_embedding_lookup.csv"


OUTPUT_DIR = DATA_DIR / "stage1_runtime_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Main data exists:", MAIN_DATA_FILE.exists())
print("Item catalog exists:", ITEM_CATALOG_FILE.exists())
print("Category embedding exists:", CATEGORY_EMBEDDING_FILE.exists())
print("Output folder:", OUTPUT_DIR)

Main data exists: True
Item catalog exists: True
Category embedding exists: True
Output folder: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts


<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: import json
# Runtime Data Loading & Initialization – Code Overview

## 1. Explanation (Precise)
- Loads **category rule artifacts** from JSON.
- Extracts `category_family_map` for runtime logic.
- Loads core datasets:
  - Main transaction data
  - Item catalog
  - Category embeddings
- Cleans column names (removes whitespace).
- Prints dataset shapes and previews data.
## 2. Possible Output
```bash
category_family_map: 45

main_df: (100000, 12)
item_catalog_df: (10000, 3)
category_embedding_df: (200, 10)

In [2]:
import json

with open(CATEGORY_RULE_FILE, "r", encoding="utf-8") as f:
    category_rule_artifacts = json.load(f)

category_family_map = category_rule_artifacts.get("category_family_map", {})

print("category_family_map:", len(category_family_map))

main_df = pd.read_csv(MAIN_DATA_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)
category_embedding_df = pd.read_csv(CATEGORY_EMBEDDING_FILE)

main_df.columns = [c.strip() for c in main_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]
category_embedding_df.columns = [c.strip() for c in category_embedding_df.columns]

print("main_df:", main_df.shape)
print("item_catalog_df:", item_catalog_df.shape)
print("category_embedding_df:", category_embedding_df.shape)

display(main_df.head())
display(item_catalog_df.head())
display(category_embedding_df.head())

main_df: (1000000, 16)
item_catalog_df: (229, 11)
category_embedding_df: (46, 9)


,transactionId,customerId,customerName,customerPersona,itemId,itemName,category,quantity,orderDate,isHoliday,isFestival,season,timeSlot,dayOfWeek,hour,month
0,1,23433,MD MOSSIN,family_grocery,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,6,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
1,1,23433,MD MOSSIN,family_grocery,453,Saad Testy Bit Salt-100gm,pantry salt,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
2,1,23433,MD MOSSIN,family_grocery,15262,Cow Brain-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
3,1,23433,MD MOSSIN,family_grocery,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
4,1,23433,MD MOSSIN,family_grocery,13986,Cow Stomach-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0


,category,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,cat_emb_8
0,Bakery-Bread,-74.173568,-111.558025,-192.203051,-70.900538,-27.881714,-73.214051,24.189736,161.398282
1,Beverage-Carbonated,-85.186907,-42.566291,242.853551,7.961231,-155.819533,0.959077,-104.045821,152.333994
2,Beverage-Hot,-28.961506,-26.255853,-7.816855,264.783572,116.117895,19.741289,-79.332576,-52.988355
3,Beverage-Juice,121.029090,179.488928,-119.588266,-205.621190,-16.090835,-78.735413,-165.299904,179.731694
4,Beverage-Water,284.556475,104.078329,-57.684418,9.785665,-68.273613,-132.973486,-38.705386,-119.340987


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: required_main_cols = [
# Main Data Validation & Cleaning – Code Overview

## 1. Explanation (Precise)
- Validates required columns in `main_data.csv`.
- Raises error if any column is missing.
- Cleans and converts data types:
  - IDs → integer
  - Text fields → trimmed strings
  - Dates → datetime format
- Ensures `weekOfMonth` exists (derives from `orderDate` if missing).
- Prepares clean data for runtime inference.
## 2. Possible Output

```bash
Data cleaning completed


In [3]:
required_main_cols = [
    "transactionId",
    "customerId",
    "itemId",
    "itemName",
    "category",
    "orderDate",
    "season",
    "isHoliday",
    "isFestival",
    "timeSlot"
]

missing_cols = [c for c in required_main_cols if c not in main_df.columns]

if missing_cols:
    raise ValueError(f"main_data.csv missing columns: {missing_cols}")

main_df = main_df.copy()
item_catalog_df = item_catalog_df.copy()

main_df["transactionId"] = main_df["transactionId"].astype(int)
main_df["customerId"] = main_df["customerId"].astype(int)
main_df["itemId"] = main_df["itemId"].astype(int)
main_df["itemName"] = main_df["itemName"].astype(str).str.strip()
main_df["category"] = main_df["category"].astype(str).str.strip()
main_df["orderDate"] = pd.to_datetime(main_df["orderDate"], errors="coerce")

item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["itemName"] = item_catalog_df["itemName"].astype(str).str.strip()
item_catalog_df["category"] = item_catalog_df["category"].astype(str).str.strip()

if "weekOfMonth" not in main_df.columns:
    main_df["weekOfMonth"] = main_df["orderDate"].apply(lambda dt: ((dt.day - 1) // 7) + 1)

main_df["weekOfMonth"] = main_df["weekOfMonth"].astype(int)

print("Data cleaning completed")

Data cleaning completed


<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: item_to_name = dict(
# Item Mapping Creation – Code Overview

## 1. Explanation (Precise)
- Creates lookup dictionaries:
  - `itemId → itemName`
  - `itemId → category`
- Enables fast access to item metadata during runtime.
- Displays sample mappings for verification.
## 2. Possible Output

```bash
item_to_name: 10000
item_to_category: 10000


In [4]:
item_to_name = dict(
    zip(
        item_catalog_df["itemId"],
        item_catalog_df["itemName"]
    )
)

item_to_category = dict(
    zip(
        item_catalog_df["itemId"],
        item_catalog_df["category"]
    )
)

print("item_to_name:", len(item_to_name))
print("item_to_category:", len(item_to_category))

sample_items = list(item_to_name.items())[:10]
sample_items

item_to_name: 229
item_to_category: 229


[(13, 'Lemon Bright Dish Wash 250ml B2 G1'),
 (27, 'Pran Toast-250g'),
 (109, 'Wheel 2in1 Clean & Fresh Powder-1Kg'),
 (125, 'Wheel Laundry Soap 125g'),
 (129, 'Vim Bar Lemon-125gm'),
 (133, 'Rin Advanced-500gm'),
 (292, 'Super White Powder-1000g'),
 (296, 'Chaka Advanced Ball Soap-125gm'),
 (318, 'Musur Pulse Kangaroo -1Kg (sp)'),
 (332, 'ACI Pure Salt-1Kg')]

<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: embedding_cols = [
# Category Embedding Mapping – Code Overview

## 1. Explanation (Precise)
- Identifies embedding columns (`cat_emb_*`).
- Validates that embedding columns exist.
- Builds a dictionary:
  - `category → embedding vector`
- Converts embedding values to numeric arrays.
- Prepares embeddings for similarity-based scoring.


## 2. Possible Output
```bash
category_to_vector: 200
Embedding dimension: 16


In [5]:
embedding_cols = [
    col for col in category_embedding_df.columns
    if col.startswith("cat_emb_")
]

if not embedding_cols:
    raise ValueError("category_embedding_lookup.csv has no cat_emb columns")

category_to_vector = {}

for _, row in category_embedding_df.iterrows():
    category = str(row["category"]).strip()
    vector = row[embedding_cols].values.astype(float)
    category_to_vector[category] = vector

print("category_to_vector:", len(category_to_vector))
print("Embedding dimension:", len(embedding_cols))

category_to_vector: 46
Embedding dimension: 8


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: item_pair_counter = Counter()
# Runtime Feature Counters & Lookup Tables – Code Overview

## 1. Explanation (Precise)
- Builds frequency-based counters from **transaction-level data**:
  - Item–item co-occurrence
  - Context-aware item frequency (season, time, etc.)
  - User–item interactions
  - User–category preferences
  - Global category popularity
- Creates fast lookup structures:
  - `pair_to_related_items`
  - `context_to_items`
  - `user_to_items`
- Enables real-time feature computation for recommendations.
## 2. Possible Output
```bash
item_pair_counter: 250000
context_item_counter: 180000
user_item_counter: 120000
user_category_counter: 80000
category_popularity_counter: 50
pair_to_related_items: 10000
context_to_items: 5000
user_to_items: 10000


In [6]:
item_pair_counter = Counter()
context_item_counter = Counter()
user_item_counter = Counter()
user_category_counter = Counter()
category_popularity_counter = Counter()

pair_to_related_items = defaultdict(Counter)
context_to_items = defaultdict(Counter)
user_to_items = defaultdict(Counter)

for transaction_id, group in main_df.groupby("transactionId", sort=False):
    item_ids = list(dict.fromkeys(group["itemId"].astype(int).tolist()))
    
    for item_i in item_ids:
        for item_j in item_ids:
            if int(item_i) == int(item_j):
                continue
            
            item_pair_counter[(int(item_i), int(item_j))] += 1
            pair_to_related_items[int(item_j)][int(item_i)] += 1

for _, row in main_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    
    item_id = int(row["itemId"])
    customer_id = int(row["customerId"])
    category = str(row["category"]).strip()
    
    context_item_counter[(context_key, item_id)] += 1
    context_to_items[context_key][item_id] += 1
    
    user_item_counter[(customer_id, item_id)] += 1
    user_to_items[customer_id][item_id] += 1
    
    user_category_counter[(customer_id, category)] += 1
    category_popularity_counter[category] += 1

print("item_pair_counter:", len(item_pair_counter))
print("context_item_counter:", len(context_item_counter))
print("user_item_counter:", len(user_item_counter))
print("user_category_counter:", len(user_category_counter))
print("category_popularity_counter:", len(category_popularity_counter))
print("pair_to_related_items:", len(pair_to_related_items))
print("context_to_items:", len(context_to_items))
print("user_to_items:", len(user_to_items))

item_pair_counter: 52198
context_item_counter: 51976
user_item_counter: 42135
user_category_counter: 8510
category_popularity_counter: 46
pair_to_related_items: 229
context_to_items: 265
user_to_items: 185


<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: customer_default_timeslot = (
# Customer Default TimeSlot Mapping – Code Overview

## 1. Explanation (Precise)
- Computes each customer’s **most frequent timeSlot** from transaction history.
- Uses groupby + value_counts to find the dominant timeSlot.
- Stores mapping:
  - `customerId → default timeSlot`
- Converts keys to integer for consistency.
- Used as fallback context when real-time timeSlot is missing.
## 2. Possible Output
```bash
customer_default_timeslot: 10000

In [7]:
customer_default_timeslot = (
    main_df
    .groupby("customerId")["timeSlot"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

customer_default_timeslot = {
    int(k): v for k, v in customer_default_timeslot.items()
}

print("customer_default_timeslot:", len(customer_default_timeslot))

list(customer_default_timeslot.items())[:10]

customer_default_timeslot: 185


[(2242, 'Night'),
 (4959, 'Night'),
 (18557, 'Noon'),
 (21212, 'Noon'),
 (21213, 'Noon'),
 (21214, 'Noon'),
 (23380, 'Noon'),
 (23381, 'Noon'),
 (23382, 'Noon'),
 (23383, 'Night')]

<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: def save_pickle(obj, file_name):
# Pickle Utility Function – Code Overview

## 1. Explanation (Precise)
- Defines a helper function to **save Python objects as pickle files**.
- Takes:
  - `obj` → object to save
  - `file_name` → output file name
- Saves the file inside `OUTPUT_DIR`.
- Prints confirmation after saving.
## 2. Possible Output
```bash
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\item_to_name.pkl


In [8]:
def save_pickle(obj, file_name):
    file_path = OUTPUT_DIR / file_name
    
    with open(file_path, "wb") as f:
        pickle.dump(obj, f)
    
    print("Saved:", file_path)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: save_pickle(item_pair_counter, "item_pair_counter.pkl")
# Runtime Artifacts Saving – Code Overview

## 1. Explanation (Precise)
- Saves all precomputed runtime data structures as **pickle files**.
- Includes:
  - Counters (co-occurrence, context, user behavior)
  - Lookup dictionaries (item, category, embeddings)
  - Mapping structures (user history, category family)
- Ensures all artifacts are stored for **fast runtime inference**.
- Prints confirmation after saving all files.
## 2. Possible Output
```bash
Saved: ...\item_pair_counter.pkl
Saved: ...\context_item_counter.pkl
Saved: ...\user_item_counter.pkl
Saved: ...\user_category_counter.pkl
Saved: ...\category_popularity_counter.pkl
Saved: ...\pair_to_related_items.pkl
Saved: ...\context_to_items.pkl
Saved: ...\user_to_items.pkl
Saved: ...\customer_default_timeslot.pkl
Saved: ...\item_to_category.pkl
Saved: ...\item_to_name.pkl
Saved: ...\category_to_vector.pkl
Saved: ...\category_family_map.pkl

All runtime artifacts saved successfully

In [9]:
save_pickle(item_pair_counter, "item_pair_counter.pkl")
save_pickle(context_item_counter, "context_item_counter.pkl")
save_pickle(user_item_counter, "user_item_counter.pkl")
save_pickle(user_category_counter, "user_category_counter.pkl")
save_pickle(category_popularity_counter, "category_popularity_counter.pkl")

save_pickle(pair_to_related_items, "pair_to_related_items.pkl")
save_pickle(context_to_items, "context_to_items.pkl")
save_pickle(user_to_items, "user_to_items.pkl")

save_pickle(customer_default_timeslot, "customer_default_timeslot.pkl")

save_pickle(item_to_category, "item_to_category.pkl")
save_pickle(item_to_name, "item_to_name.pkl")
save_pickle(category_to_vector, "category_to_vector.pkl")
save_pickle(category_family_map, "category_family_map.pkl")

print("All runtime artifacts saved successfully")

Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\item_pair_counter.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\context_item_counter.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\user_item_counter.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\user_category_counter.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\category_popularity_counter.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\pair_to_related_items.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\context_to_items.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\user_to_items.pkl
Saved: C:\D drive\item_recommendation_model_create\data\stage1_runtime_artifacts\customer_default_timeslot.pkl
Saved: C:\D drive\item_recommendation_model_creat

<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: expected_files = [
# Runtime Artifacts Verification – Code Overview

## 1. Explanation (Precise)
- Lists all expected runtim
## 2. Possible Output


```bash
item_pair_counter.pkl exists: True
context_item_counter.pkl exists: True
user_item_counter.pkl exists: True
user_category_counter.pkl exists: True
category_popularity_counter.pkl exists: True
pair_to_related_items.pkl exists: True
context_to_items.pkl exists: True
user_to_items.pkl exists: True
customer_default_timeslot.pkl exists: True
item_to_category.pkl exists: True
item_to_name.pkl exists: True
category_to_vector.pkl exists: True
category_family_map.pkl exists: True


In [10]:
expected_files = [
    "item_pair_counter.pkl",
    "context_item_counter.pkl",
    "user_item_counter.pkl",
    "user_category_counter.pkl",
    "category_popularity_counter.pkl",
    "pair_to_related_items.pkl",
    "context_to_items.pkl",
    "user_to_items.pkl",
    "customer_default_timeslot.pkl",
    "item_to_category.pkl",
    "item_to_name.pkl",
    "category_to_vector.pkl",
    "category_family_map.pkl"
]

for file_name in expected_files:
    file_path = OUTPUT_DIR / file_name
    print(file_name, "exists:", file_path.exists())

item_pair_counter.pkl exists: True
context_item_counter.pkl exists: True
user_item_counter.pkl exists: True
user_category_counter.pkl exists: True
category_popularity_counter.pkl exists: True
pair_to_related_items.pkl exists: True
context_to_items.pkl exists: True
user_to_items.pkl exists: True
customer_default_timeslot.pkl exists: True
item_to_category.pkl exists: True
item_to_name.pkl exists: True
category_to_vector.pkl exists: True


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: with open(OUTPUT_DIR / "item_to_name.pkl", "rb") as f:
# Runtime Artifact Loading Test – Code Overview

## 1. Explanation (Precise)
- Loads saved pickle files:
  - `item_to_name`
  - `pair_to_related_items`
- Verifies that artifacts can be correctly reloaded.
- Prints sizes and sample data for validation.

## 2. Possible Output

### ✅ Console Output:
```bash
Loaded item_to_name: 10000
Loaded pair_to_related_items: 10000

In [11]:
with open(OUTPUT_DIR / "item_to_name.pkl", "rb") as f:
    loaded_item_to_name = pickle.load(f)

with open(OUTPUT_DIR / "pair_to_related_items.pkl", "rb") as f:
    loaded_pair_to_related_items = pickle.load(f)

print("Loaded item_to_name:", len(loaded_item_to_name))
print("Loaded pair_to_related_items:", len(loaded_pair_to_related_items))

print("Sample item name:", list(loaded_item_to_name.items())[:5])

Loaded item_to_name: 229
Loaded pair_to_related_items: 229
Sample item name: [(13, 'Lemon Bright Dish Wash 250ml B2 G1'), (27, 'Pran Toast-250g'), (109, 'Wheel 2in1 Clean & Fresh Powder-1Kg'), (125, 'Wheel Laundry Soap 125g'), (129, 'Vim Bar Lemon-125gm')]


<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: Code cell execution step
**Explanation:** This cell executes logic related to `Code cell execution step`.

**Possible Output:** Direct output may not appear; variables/state can update silently.
